# Modélisation – [Hypothèse : code et intitulé]
**Contributeurs :** [prénom] · **Date :** [jj/mm/2026]

Notebook construit à partir du template commun de l'équipe. Il passe par `commun.py` pour tout ce qui doit être identique d'une hypothèse à l'autre : la table, le masque des lignes, les fenêtres, la naïve, `evaluer` et `tester`.

**Question :** [la question de l'hypothèse]

> **Mode d'emploi du template (à supprimer dans le notebook final)** : adapter la question, la section 0 (blocs de variables) et la section 4 (modèles). Relancer le notebook depuis la section 3 avant de relancer des modèles. Les sections 5 à 7 se lisent avec leurs propres résultats ; les textes « Comment lire » sont à compléter avec un exemple tiré des tableaux.

## 0. Paramètres de l'hypothèse
- `HYPOTHESE` : préfixe du nom des modèles dans les tableaux (« H3b – Lasso – bloc complet »).
- Les blocs de variables, dans l'ordre où ils entrent dans les modèles ; `VARIABLES_BLOC` les réunit pour le constat rapide.
- `LOG` et `LOG1P` : les variables asymétriques de la table (audit complémentaire, règle : asymétrie > 1), passées en log, ou en log(1 + x) si elles contiennent des zéros, pour les seuls modèles qui en ont besoin. `prix_n1` passe en log avec la cible, pour que le modèle compare log(prix) à log(prix N-1).

In [ ]:
HYPOTHESE = "H3x"

# Blocs de variables (EXEMPLE : à remplacer par ceux de l'hypothèse)
BLOC_1 = ["taux_vacance", "part_avant1946"]
VARIABLES_BLOC = BLOC_1                         # tous les blocs réunis (constat rapide, section 2)

# Variables asymétriques de la table (audit complémentaire, version 2)
LOG = ["nb_ventes_n1", "densite_population", "niveau_vie_median",
       "taux3_vols_sans_violence", "taux3_vols_vehicules",
       "taux3_violences_sexuelles", "taux3_cambriolages", "taux3_violences_intrafamiliales",
       "taux_equip_commerces", "taux_equip_services", "taux_equip_sante", "taux_equip_sports",
       "part_sans_voiture", "part_moins40m2", "part_rsec", "part_apparts",
       "part_familles_monoparentales_pct_fam"]
LOG1P = ["taux3_vols_violents", "taux3_stupefiants", "taux_equip_tourisme", "taux_equip_transports"]

# Variables de l'hypothèse seulement ; prix_n1 passe en log avec la cible
VARIABLES_LOG = [v for v in LOG if v in VARIABLES_BLOC] + ["prix_n1"]
VARIABLES_LOG1P = [v for v in LOG1P if v in VARIABLES_BLOC]
print("log :", VARIABLES_LOG, "| log(1 + x) :", VARIABLES_LOG1P)

## 1. Import et chargement
`charger_table()` retire une seule fois les lignes à case vide (masque commun) : tous les modèles et la naïve sont jugés sur les mêmes lignes. `fenetres()` donne une fenêtre par année jugée (2023 et 2024), avec l'apprentissage sur les années d'avant.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from copy import deepcopy
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import commun
from commun import CIBLE, charger_table, fenetres, plis_temporels, evaluer, tester
from gabarit_hierarchique import ModeleHierarchique
from gabarit_boosting import BoostingEvolution, prevoir_boosting

table = charger_table()
FENETRES = fenetres(table)

absentes = sorted(set(VARIABLES_BLOC + VARIABLES_LOG + VARIABLES_LOG1P) - set(table.columns))
assert not absentes, f"Variables absentes de la table : {absentes}"
print("Table :", table.shape)
for a, (app, ev) in FENETRES.items():
    print(f"Fenêtre {a} : apprentissage {sorted(app['annee'].unique())} ({len(app)} lignes) | jugée {a} ({len(ev)} lignes)")

## 2. Constat rapide
Distribution des variables du bloc et lien avec le prix (rho de Spearman), sur les années d'apprentissage de la première fenêtre seulement : l'année jugée n'est jamais regardée avant la modélisation.

In [ ]:
app_constat = FENETRES[commun.ANNEES_JUGEES[0]][0]
print(app_constat[VARIABLES_BLOC].describe().round(2).to_string())

liens = []
for type_bien, e in app_constat.groupby("type_bien"):
    for v in VARIABLES_BLOC:
        liens.append({"type de bien": type_bien, "variable": v, "rho": stats.spearmanr(e[v], e[CIBLE]).statistic})
pd.DataFrame(liens).pivot(index="variable", columns="type de bien", values="rho").round(2)

### 2.2 Redondances dans le bloc et lien avec l'écart au prix N-1
- **Redondances** : couples de variables du bloc dont |rho| dépasse 0,7 ; deux variables aussi liées portent la même information.
- **Écart au prix N-1** : `prix_n1` porte déjà le niveau du prix. Ce que le bloc peut apporter, c'est ce que `prix_n1` n'explique pas : l'évolution relative prix / prix N-1 − 1. On mesure son lien avec chaque variable (rho de Spearman).
- Années d'apprentissage de la première fenêtre seulement. Un même territoire revient chaque année : les rho décrivent, ils ne se testent pas.

In [ ]:
couples = []
for type_bien, e in app_constat.groupby("type_bien"):
    rho = e[VARIABLES_BLOC].corr(method="spearman")
    for i, a in enumerate(VARIABLES_BLOC):
        for b in VARIABLES_BLOC[i + 1:]:
            if abs(rho.loc[a, b]) > 0.7:
                couples.append({"type de bien": type_bien, "variable 1": a, "variable 2": b, "rho": round(rho.loc[a, b], 2)})
print(pd.DataFrame(couples).to_string(index=False) if couples else "Aucun couple au-delà de 0,7")

evolution_app = app_constat[CIBLE] / app_constat["prix_n1"] - 1
liens_evolution = []
for type_bien, e in app_constat.groupby("type_bien"):
    for v in VARIABLES_BLOC:
        liens_evolution.append({"type de bien": type_bien, "variable": v,
                                "rho": stats.spearmanr(e[v], evolution_app.loc[e.index]).statistic})
pd.DataFrame(liens_evolution).pivot(index="variable", columns="type de bien", values="rho").round(2)

## 3. Fonctions du notebook
Chaque modèle est lancé en une ligne : `lancer(nom, modèle, variables, log=..., grille=...)`.
- `nom` : le nom du modèle dans les tableaux (le préfixe de l'hypothèse est ajouté automatiquement) ;
- `modèle` : le modèle à utiliser (régression linéaire, Lasso, gradient boosting `BoostingEvolution()`, modèle hiérarchique) ;
- `variables` : les colonnes de la table données au modèle ;
- `log=True` : la cible et les variables de `VARIABLES_LOG` passent en log, celles de `VARIABLES_LOG1P` en log(1 + x) ; à réserver aux modèles qui en ont besoin (régressions, Lasso, modèle hiérarchique), jamais aux arbres ;
- `grille` : les valeurs d'un réglage à tester (par exemple `alpha` pour le Lasso).

Ce que fait `lancer`, pour chacune des deux années jugées (2023, puis 2024) et pour chaque type de bien :
1. elle prend les années d'apprentissage, avant l'année jugée ;
2. elle met les variables à l'échelle (sauf pour le gradient boosting et le modèle hiérarchique) et entraîne le modèle ; avec une grille, elle choisit d'abord le meilleur réglage, chaque année d'apprentissage servant tour à tour de validation (`plis_temporels` de `commun.py`) ;
3. elle prévoit le prix de l'année jugée, remis en €/m² si le modèle a appris en log ;
4. elle compare la prévision au prix réel et à la naïve (prix N = prix N-1) avec `evaluer` et `tester` de `commun.py`.

Elle affiche ensuite l'erreur du modèle (MAE, en €/m²), celle de la naïve, le gain sur la naïve (négatif : le modèle fait moins bien) et la part des sous-bassins où le modèle gagne. Elle garde les modèles entraînés (`ajustes`, pour la section 6) et leurs prévisions (`previsions`, pour la section 5.2). Relancer une cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois.

Le modèle hiérarchique (`gabarit_hierarchique.py`) passe par une enveloppe : il n'est pas mis dans le pipeline (il met lui-même ses variables à l'échelle) et n'accepte pas de grille.

Le gradient boosting (`BoostingEvolution` de `gabarit_boosting.py`, gabarit commun P3) passe par `prevoir_boosting` : ni mise à l'échelle ni log, réglage avec la grille commune (`GRILLE_REFERENCE`) sur les plis temporels quand `grille` n'est pas donnée (`grille={}` : pas de réglage). Il apprend l'évolution log(prix / prix N-1) et rend un prix en €/m² (prix N-1 × évolution prévue) : la cible jugée reste le prix. `prix_n1` doit être dans les variables.

In [ ]:
resultats, tests, ajustes, previsions = [], [], {}, {}


def en_log(X):
    """Variables asymétriques en log ; log(1 + x) pour celles qui ont des zéros."""
    X = X.copy()
    cols = [c for c in VARIABLES_LOG if c in X.columns]
    cols1p = [c for c in VARIABLES_LOG1P if c in X.columns]
    assert (X[cols] > 0).all().all(), "valeur nulle ou négative dans une variable de VARIABLES_LOG : la passer dans LOG1P"
    X[cols] = np.log(X[cols])
    X[cols1p] = np.log1p(X[cols1p])
    return X


class Hierarchique(RegressorMixin, BaseEstimator):
    """Enveloppe scikit-learn du modèle hiérarchique : log de la cible si demandé, prévisions en €/m²."""
    def __init__(self, modele=None, log=False):
        self.modele = modele
        self.log = log

    def fit(self, X, y):
        self.modele_ = deepcopy(self.modele).fit(X, np.log(y) if self.log else y)
        return self

    def predict(self, X):
        p = np.asarray(self.modele_.predict(X))
        return np.exp(p) if self.log else p


def lancer(nom, modele, variables, log=False, grille=None):
    """Ajuste un modèle par fenêtre et par type de bien, puis evaluer et tester ; affiche le résultat."""
    nom = f"{HYPOTHESE} – {nom}"
    # Relancer une cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois
    resultats[:] = [r for r in resultats if r["modèle"] != nom]
    tests[:] = [t for t in tests if t["modèle"] != nom]
    for annee, (app, ev) in FENETRES.items():
        if isinstance(modele, BoostingEvolution):
            # Gabarit commun P3 : réglage, un modèle par type de bien et prévision en €/m² faits par prevoir_boosting
            assert not log, "pas de log pour le boosting : le gabarit transforme lui-même la cible"
            prevision, modeles = prevoir_boosting(app, ev, variables, cible=modele.cible, grille=grille)
            for type_bien, m in modeles.items():
                idx = ev.index[ev.index.get_level_values("type_bien") == type_bien]
                ajustes[(nom, annee, type_bien)] = (m, ev.loc[idx, variables], ev.loc[idx, CIBLE])
        else:
            prevision = pd.Series(np.nan, index=ev.index)
            for type_bien, e in app.groupby("type_bien"):
                idx = ev.index[ev.index.get_level_values("type_bien") == type_bien]
                X_app, X_ev = e[variables], ev.loc[idx, variables]
                if isinstance(modele, ModeleHierarchique):
                    assert not grille, "pas de grille pour le modèle hiérarchique"
                    if log:
                        X_app, X_ev = en_log(X_app), en_log(X_ev)
                    m = Hierarchique(modele, log).fit(X_app, e[CIBLE])
                    prevision.loc[idx] = m.predict(X_ev)
                    ajustes[(nom, annee, type_bien)] = (m, X_ev, ev.loc[idx, CIBLE])
                    continue
                m = Pipeline([("echelle", StandardScaler()), ("modele", clone(modele))])
                prefixe = "modele__"
                if log:
                    X_app, X_ev = en_log(X_app), en_log(X_ev)
                    m = TransformedTargetRegressor(regressor=m, func=np.log, inverse_func=np.exp)
                    prefixe = "regressor__modele__"
                if grille:
                    # plis recalculés sur e (positions dans e, même ordre que X_app)
                    m = GridSearchCV(m, {prefixe + k: v for k, v in grille.items()},
                                     cv=plis_temporels(e), scoring="neg_mean_absolute_error")
                m.fit(X_app, e[CIBLE])
                prevision.loc[idx] = m.predict(X_ev)
                ajustes[(nom, annee, type_bien)] = (m, X_ev, ev.loc[idx, CIBLE])
        assert prevision.notna().all(), f"{nom} : prévisions manquantes"
        previsions[(nom, annee)] = prevision          # gardée pour comparer les modèles entre eux (section 5.2)
        evaluer(nom, ev, prevision, resultats)
        tester(nom, ev, prevision, tests)

    # Résultat du modèle, affiché tout de suite : erreur du modèle, erreur de la naïve, gain
    res = pd.DataFrame(resultats)
    res = res[res["modèle"] == nom][["type de bien", "année jugée", "MAE (€/m²)", "MAE naïve (€/m²)",
                                     "gain sur la naïve (%)", "bassins où le modèle gagne (%)"]]
    print(nom, f"({len(variables)} variables)")
    print(res.round(1).to_string(index=False))
    return nom

## 4. Modèles
Du plus simple au plus complexe ; chaque étape ajoute un bloc à la précédente. Tous les modèles sont gardés, y compris ceux qui perdent. Les régressions et le Lasso sont faits en log, la référence aussi, pour une comparaison à égalité ; le gradient boosting n'a pas besoin du log : le gabarit transforme lui-même la cible (section 4.4).

In [ ]:
# Blocs successifs : chaque étape ajoute un bloc à la précédente (EXEMPLE : à adapter)
REFERENCE_VARS = ["prix_n1"]
BLOC_COMPLET = REFERENCE_VARS + BLOC_1

### 4.1 Référence : prix N-1 seul
La naïve (prix N = prix N-1) est déjà dans chaque ligne du tableau (« MAE naïve »). Le modèle « prix N-1 seul » est la référence que le bloc doit battre (section 5.2).

In [ ]:
REFERENCE = lancer("Régression linéaire – prix N-1 seul", LinearRegression(), REFERENCE_VARS, log=True)

### 4.2 Régression linéaire : prix N-1 + bloc

In [ ]:
COMPLET = lancer("Régression linéaire – bloc complet", LinearRegression(), BLOC_COMPLET, log=True)

### 4.3 Lasso : quelles variables survivent
Le Lasso ramène à zéro les coefficients des variables qui n'apportent rien. `alpha` est réglé sur les plis temporels. Les valeurs testées sont petites : la cible est en log et les variables sont mises à l'échelle, les coefficients sont donc petits eux aussi.

In [ ]:
LASSO = lancer("Lasso – bloc complet", Lasso(max_iter=10_000), BLOC_COMPLET, log=True,
               grille={"alpha": [0.0001, 0.001, 0.01, 0.1]})

### 4.4 Gradient boosting (gabarit commun P3)
Gabarit de l'équipe (`gabarit_boosting.py`) : même modèle, même grille et même réglage pour toutes les hypothèses, ce qui rend les boostings comparables d'une hypothèse à l'autre. Le modèle apprend l'évolution du prix par rapport au prix N-1 et rend une prévision en €/m² : la cible jugée reste le prix. Un arbre qui prévoirait directement le prix rendrait la moyenne des prix de chaque feuille et perdrait la précision du prix N-1 de chaque sous-bassin. Pas de mise à l'échelle ni de log : un arbre n'en a pas besoin. `prix_n1` doit être dans les variables (il l'est dans `BLOC_COMPLET`).

In [ ]:
BOOSTING = lancer("Gradient boosting – bloc complet", BoostingEvolution(), BLOC_COMPLET)

### 4.5 (option) Modèle hiérarchique
Une ordonnée à l'origine propre à chaque sous-bassin (`gabarit_hierarchique.py`). Avec `prix_n1`, l'effet du territoire est presque nul (ICC proche de 0, voir `test.py`). Retirer le `#` pour le lancer.

In [ ]:
# lancer("Hiérarchique – bloc complet", ModeleHierarchique(), BLOC_COMPLET, log=True)

## 5. Mesures et tests
Trois tableaux, du plus simple au plus exigeant :
1. **`res` : les erreurs.** MAE du modèle et de la naïve en €/m², gain sur la naïve en %, part des sous-bassins où le modèle gagne. C'est une description : un modèle peut avoir une MAE plus basse par hasard.
2. **`tst` : les tests contre la naïve.** Ils disent si l'écart avec la naïve est plus grand que le hasard.
   - **Écart de MAE avec la naïve** : erreur du modèle moins erreur de la naïve, en €/m² (négatif : le modèle fait mieux).
   - **IC 95 %** (bootstrap par bassin de vie) : la fourchette dans laquelle se trouve cet écart, en tenant compte du hasard. Si toute la fourchette est sous 0, le modèle fait vraiment mieux ; si elle contient 0, on ne peut pas conclure ; si elle est toute au-dessus de 0, il fait vraiment moins bien.
   - **p-value de Wilcoxon** : le modèle fait-il mieux (ou moins bien) dans la plupart des sous-bassins ? Avec plusieurs centaines de sous-bassins, elle est presque toujours très petite : on la lit avec la taille d'effet.
   - **Rang-bisériale** (taille d'effet) : de −1 (le modèle fait mieux partout) à +1 (il fait moins bien partout) ; autour de 0, pas de différence nette.
3. **`decision` : la règle de décision.** Une année est réussie si l'IC est entièrement sous 0, la rang-bisériale négative et le gain d'au moins 3 %. Un modèle est retenu s'il réussit la majorité des années jugées (ici, les deux).

In [ ]:
res = pd.DataFrame(resultats)
res.round(2)

In [ ]:
tst = pd.DataFrame(tests)
tst.round(3)

### 5.1 Règle de décision contre la naïve

In [ ]:
SEUIL_GAIN = 3      # % de gain sur la MAE de la comparaison, fixé avant les résultats


def regle_decision(res, tst, comparaison="naïve"):
    """Années réussies et décision, par modèle et type de bien."""
    cles = ["modèle", "type de bien", "année jugée"]
    regle = res[cles + [f"gain sur la {comparaison} (%)"]].merge(
        tst[cles + ["IC 95 % haut (€/m²)", "rang-bisériale"]], on=cles)
    regle["année réussie"] = ((regle["IC 95 % haut (€/m²)"] < 0) & (regle["rang-bisériale"] < 0)
                              & (regle[f"gain sur la {comparaison} (%)"] >= SEUIL_GAIN))
    decision = regle.groupby(["modèle", "type de bien"], sort=False)["année réussie"].agg(["sum", "count"])
    decision["retenu"] = decision["sum"] > decision["count"] / 2
    return decision.rename(columns={"sum": "années réussies", "count": "années jugées"})


regle_decision(res, tst)

**Comment lire le tableau des tests (exemple à compléter avec une ligne de `tst`) :** …

**Résultats :** …

### 5.2 Le bloc apporte-t-il quelque chose au prix N-1 seul ?
La question propre à l'hypothèse : les variables du bloc améliorent-elles la prévision **par rapport au modèle « prix N-1 seul »** ?

On applique exactement les mêmes mesures et la même règle de décision, en remplaçant la naïve par la prévision du modèle « prix N-1 seul ». Les fonctions `evaluer` et `tester` lisent la prévision de comparaison dans la colonne `prix_n1` : on leur donne une copie de l'année jugée où cette colonne contient la prévision de la référence.

Lecture : un gain positif et un intervalle du bootstrap entièrement sous 0 veulent dire que le modèle fait vraiment mieux que la référence.

In [ ]:
resultats_ref, tests_ref = [], []
modeles_compares = [n for n in dict.fromkeys(k[0] for k in previsions) if n != REFERENCE]

for nom in modeles_compares:
    for annee, (app, ev) in FENETRES.items():
        ev_ref = ev.copy()
        ev_ref["prix_n1"] = previsions[(REFERENCE, annee)]     # la comparaison se fait avec la référence
        evaluer(nom, ev_ref, previsions[(nom, annee)], resultats_ref)
        tester(nom, ev_ref, previsions[(nom, annee)], tests_ref)

# Mêmes tableaux que contre la naïve, avec « référence » à la place de « naïve »
res_ref = pd.DataFrame(resultats_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
tst_ref = pd.DataFrame(tests_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
display(res_ref[["modèle", "type de bien", "année jugée", "MAE (€/m²)", "MAE référence (€/m²)",
                 "gain sur la référence (%)", "bassins où le modèle gagne (%)"]].round(2))
display(tst_ref[["modèle", "type de bien", "année jugée", "écart de MAE avec la référence (€/m²)",
                 "IC 95 % bas (€/m²)", "IC 95 % haut (€/m²)", "p-value Wilcoxon", "rang-bisériale"]].round(3))

#### Règle de décision contre la référence
Même règle que contre la naïve, avec le modèle « prix N-1 seul » à la place de la naïve.

In [ ]:
regle_decision(res_ref, tst_ref, comparaison="référence")

**Comment lire le tableau des tests (exemple à compléter) :** …

**Résultats contre le modèle « prix N-1 seul » :** …

## 6. Interprétation

### 6.1 Contexte : le marché s'est-il retourné ?
Les modèles apprennent sur les années de hausse 2020-2022, puis sont jugés sur 2023 et 2024. Pour lire leurs résultats, on mesure l'évolution du prix d'une année sur l'autre (prix / prix N-1 − 1), en moyenne et en médiane, par année et par type de bien, jusqu'à 2024. L'année 2025, réservée au test final, n'est pas regardée.

In [ ]:
jusqua_2024 = table[table["annee"] <= commun.ANNEE_VALIDATION]
evolution = (jusqua_2024[CIBLE] / jusqua_2024["prix_n1"] - 1) * 100
retournement = (evolution.groupby([jusqua_2024["annee"], jusqua_2024.index.get_level_values("type_bien")])
                .agg(["mean", "median"]).round(1)
                .rename(columns={"mean": "évolution moyenne (%)", "median": "évolution médiane (%)"}))
retournement.unstack()

### 6.2 Importance des variables
Importance par permutation, sur l'année jugée : on mélange au hasard les valeurs d'une variable entre les sous-bassins, 10 fois, et on mesure de combien l'erreur du modèle change (moyenne et écart-type). Une hausse : la variable était utile ; 0 : le modèle ne s'en sert pas ; une baisse : sur cette année, la variable dessert le modèle. Une variable compte si sa valeur dépasse nettement deux écarts-types. Choisir le modèle, l'année jugée et le type de bien.

In [ ]:
choix = (LASSO, 2024, "maison")        # modèle, année jugée, type de bien étudiés (à adapter)
m, X_ev, y_ev = ajustes[choix]
imp = permutation_importance(m, X_ev, y_ev, scoring="neg_mean_absolute_error",
                             n_repeats=10, random_state=commun.GRAINE)
pd.DataFrame({"hausse de la MAE (€/m²)": imp.importances_mean,
              "écart-type (€/m²)": imp.importances_std},
             index=X_ev.columns).sort_values("hausse de la MAE (€/m²)", ascending=False).round(1)

### 6.3 Analyse des erreurs selon le contrôle qualité
L'erreur est-elle plus grande là où les données sont estimées ou complétées ? Pour chaque variable de contrôle qualité : groupe « part nulle », puis quartiles parmi les parts non nulles. Par groupe : erreur moyenne en €/m² et en % du prix (l'erreur en €/m² grandit avec le niveau des prix), et erreur moyenne signée (positive : le modèle surestime). Rho de Spearman entre la variable et l'erreur, en €/m² et en %. Même modèle qu'en 6.2.

In [ ]:
CONTROLE = ["part_indicateurs_estimes", "part_logements_taux_impute", "part_menages_couverts"]

m, X_ev, y_ev = ajustes[choix]
erreur = pd.Series(m.predict(X_ev), index=X_ev.index) - y_ev       # signée : positive = prix surestimé
erreur_pct = erreur.abs() / y_ev * 100                             # erreur absolue en % du prix réel
controle = table.loc[X_ev.index, CONTROLE]

for v in CONTROLE:
    x = controle[v]
    groupe = pd.Series("part nulle", index=x.index)
    non_nul = x > 0
    if non_nul.sum() >= 4:
        groupe[non_nul] = pd.qcut(x[non_nul], 4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop").astype(str)
    tableau = pd.DataFrame({"MAE (€/m²)": erreur.abs(), "erreur en % du prix": erreur_pct,
                            "erreur signée (€/m²)": erreur}).groupby(groupe).mean()
    tableau["lignes"] = groupe.value_counts()
    rho, p = stats.spearmanr(x, erreur.abs())
    rho_pct, _ = stats.spearmanr(x, erreur_pct)
    print(f"\n{v} : rho avec l'erreur en €/m² = {rho:.2f} (p = {p:.3f}) | rho avec l'erreur en % = {rho_pct:.2f}")
    print(tableau.round(1).to_string())

### 6.4 et suivantes : analyses propres à l'hypothèse

## 7. Conclusion
- Contre la naïve : modèles retenus ou non, par type de bien, avec les chiffres (gain, intervalle, rang-bisériale).
- Contre le modèle « prix N-1 seul » : le bloc apporte-t-il quelque chose, et quels blocs portent l'apport.
- Variables qui portent l'effet (Lasso, importance).
- Erreurs concentrées ou non dans les territoires aux données estimées.
- Réponses aux questions propres à l'hypothèse (sections 6.4 et suivantes).
- Limites.